In [38]:
#%pip install faiss-cpu
#%pip install langchain
#%pip install langchain_core
#%pip install -U langchain-community
%pip install sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
Note: you may need to restart the kernel to use updated packages.


In [31]:
%pip install langchain==0.3.27

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.1.0
    Uninstalling langchain-core-1.1.0:
      Successfully uninstalled langchain-core-1.1.0
  Attempting uninstall: langchain
    Found existing installation: langchain 1.0.0
    Uninstalling langchain-1.0.0:
      Successfully uninstalled langchain-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [132]:
from langchain_core.documents import Document
from langchain.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
import langchain
langchain.debug = False
from langchain.vectorstores.faiss import FAISS  # si usas vectorstore FAISS de LangChain
from langchain.embeddings import HuggingFaceEmbeddings  # para usar tu modelo PlanTL-GOB-ES
import os
import pandas as pd

# --- Función para cargar los documentos CANTEMIST como Document de LangChain
def load_cantemist_langchain(base_dir="../data/cantemist"):
    docs = []
    subsets = ["train-set", "dev-set1", "dev-set2", "test-set"]
    for subset in subsets:
        txt_folder = os.path.join(base_dir, subset, "cantemist-coding", "txt")
        if not os.path.isdir(txt_folder):
            continue
        for fname in os.listdir(txt_folder):
            if not fname.endswith(".txt"):
                continue
            path = os.path.join(txt_folder, fname)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            metadata = {"source": f"{subset}/{fname}"}
            docs.append(Document(page_content=text, metadata=metadata))
    return docs

# --- Definir splitters
# Parent splitter: divide en pedazos grandes — podrías usar documento completo, o párrafos largos
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20000,  # tamaño “padre”
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

# Child splitter: fragmentos pequeños para embedding
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,
    add_start_index=True
)

# --- Embeddings con PlanTL-GOB-ES
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    model_kwargs = {'device': 'cuda'}
)
# --- VectorStore FAISS
# Usamos FAISS desde langchain para indexar los fragmentos “hijos”
vectorstore = FAISS.from_documents(
    documents=child_splitter.split_documents(load_cantemist_langchain()),
    embedding=embeddings
)

# --- Docstore para los padres
from langchain.storage import InMemoryStore
docstore = InMemoryStore()

# --- Agregar los documentos (padres) al retriever
docs = load_cantemist_langchain()

In [133]:
# --- Construir el ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    search_type="similarity",
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter, search_kwargs = {'k':10,  "fetch_k": 5}
)

retriever.add_documents(docs)


In [134]:

# --- Ahora podemos hacer consultas
query = "leiomioma"
# Esto va a usar los embeddings de los hijos, pero devolverá textos “padres”
results = retriever.invoke(query)

for doc in results:
    print("--- Padre recuperado ---")
    print("Texto:", doc.page_content, "…")
    print("Metadata:", doc.metadata)
    print("\n")


--- Padre recuperado ---
Texto: Anamnesis
Mujer de 49 años remitida en octubre de 2018 desde Atención Primaria a consultas externas de Cirugía General, por aparición de nódulo palpable en mama derecha, de un mes de evolución.
En ese momento, la paciente no presentaba antecedentes medicoquirúrgicos de interés, ni refería hábitos tóxicos. Trabajaba como ama de casa y tenía buen apoyo familiar. Como antecedentes familiares, una tía paterna fue diagnosticada de carcinoma de mama a los 65 años.
Como tratamiento crónico, se había pautado recientemente acetato de medroxiprogesterona por hiperplasia endometrial simple.

Exploración física
La paciente presentaba buen estado general, con un performance status (PS) 0. En la exploración física, se palpaba un nódulo sólido, de aproximadamente 4 cm en cuadrante superoexterno de mama derecha. La exploración axilar resultó dentro de la normalidad, así como el resto de la exploración física.

Pruebas complementarias
Se solicitó mamografía/ecografía, qu

# Pipeline RAG con ParentDocumentRetriever en LangChain para CANTEMIST

Este documento describe un pipeline para construir un **RAG (Retrieval-Augmented Generation)** usando LangChain, con embeddings en GPU, FAISS como vectorstore, y el modelo **PlanTL-GOB-ES**. El objetivo es recuperar documentos médicos completos (padres) usando embeddings de fragmentos más pequeños (hijos).

---

## 1. Dependencias

Se usan los siguientes paquetes:

```python
from langchain_core.documents import Document
from langchain.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.vectorstores.faiss import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
import os
import pandas as pd
import langchain
langchain.debug = False
```

> `langchain.debug = False` desactiva mensajes de debug para simplificar la salida.

---

## 2. Cargar documentos CANTEMIST

Los informes médicos se cargan como objetos `Document` de LangChain. Cada documento contiene:

* `page_content`: texto completo del informe.
* `metadata`: diccionario con información adicional, en este caso la fuente del documento.

```python
def load_cantemist_langchain(base_dir="../data/cantemist"):
    docs = []
    subsets = ["train-set", "dev-set1", "dev-set2", "test-set"]
    for subset in subsets:
        txt_folder = os.path.join(base_dir, subset, "cantemist-coding", "txt")
        if not os.path.isdir(txt_folder):
            continue
        for fname in os.listdir(txt_folder):
            if not fname.endswith(".txt"):
                continue
            path = os.path.join(txt_folder, fname)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            metadata = {"source": f"{subset}/{fname}"}
            docs.append(Document(page_content=text, metadata=metadata))
    return docs
```

---

## 3. Definir los splitters

Se utilizan **dos niveles de chunking**:

1. **Parent splitter**: divide en documentos “padre” grandes (todo el texto o párrafos largos).
2. **Child splitter**: divide los documentos padres en fragmentos más pequeños para generar embeddings.

```python
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20000,  # tamaño padre
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,    # tamaño hijo
    chunk_overlap=10,
    length_function=len,
    add_start_index=True
)
```

---

## 4. Embeddings en GPU con HuggingFace

Se usa `HuggingFaceEmbeddings` para cargar el modelo **PlanTL-GOB-ES** directamente en GPU:

```python
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",  # reemplazar por PlanTL-GOB-ES
    model_kwargs = {'device': 'cuda'}
)
```

> Esto permite que todos los embeddings se calculen en GPU, acelerando el proceso.

---

## 5. Crear el vectorstore FAISS

Los fragmentos “hijos” se indexan en FAISS para búsqueda de similitud:

```python
vectorstore = FAISS.from_documents(
    documents=child_splitter.split_documents(load_cantemist_langchain()),
    embedding=embeddings
)
```

---

## 6. Configurar el docstore de padres

Se guarda la información de los documentos padres usando un **docstore en memoria**:

```python
docstore = InMemoryStore()
docs = load_cantemist_langchain()
```

---

## 7. Construir el `ParentDocumentRetriever`

El retriever permite:

* Buscar por similitud usando embeddings de los hijos.
* Devolver los documentos padres completos.
* Configurar el número de documentos hijos a considerar (`k`) y `fetch_k` para búsqueda refinada.

```python
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    search_type="similarity",
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={'k':10,  "fetch_k": 5}
)

retriever.add_documents(docs)
```

---

## 8. Realizar consultas

Se puede consultar el retriever con cualquier término médico o frase:

```python
query = "leiomioma"
results = retriever.invoke(query)

for doc in results:
    print("--- Padre recuperado ---")
    print("Texto:", doc.page_content[:300], "…")  # mostrar solo inicio
    print("Metadata:", doc.metadata)
    print("\n")
```

> Los resultados devueltos son los **documentos padres**, aunque la búsqueda se hizo sobre los fragmentos hijos.

---

## 9. Resumen del pipeline

1. Cargar documentos CANTEMIST como `Document`.
2. Dividir documentos en **padres** y luego en **hijos**.
3. Crear embeddings de los hijos usando **PlanTL-GOB-ES en GPU**.
4. Indexar los hijos en **FAISS**.
5. Crear `ParentDocumentRetriever` para mapear hijos a padres.
6. Consultar usando `retriever.invoke(query)` y obtener los padres más relevantes.

Este enfoque permite:

* Recuperación eficiente de documentos largos.
* Uso de GPU para embeddings grandes.
* Búsqueda jerárquica (child → parent).

---

¿Quieres que haga también un **diagrama esquemático del flujo de padres e hijos** para agregar al documento? Esto ayuda mucho a visualizar el RAG.


In [135]:
import os
from collections import defaultdict
from langchain.schema import Document
import pandas as pd
from tqdm import tqdm
# ------------------------
# 1) Función para extraer morfologías desde archivos .ann
# ------------------------
def extract_morfologias_from_ann(base_dir="../data/cantemist"):
    """
    Recorre train/dev/test, lee los .ann y devuelve un set de morfologías únicas.
    """
    morfologias_set = set()
    
    subsets = ["train-set", "dev-set1", "dev-set2", "test-set"]
    for subset in subsets:
        ner_folder = os.path.join(base_dir, subset, "cantemist-ner")
        if not os.path.exists(ner_folder):
            continue

        for fname in os.listdir(ner_folder):
            if not fname.endswith(".ann"):
                continue
            ann_path = os.path.join(ner_folder, fname)
            with open(ann_path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) != 3:
                        continue
                    tag_info, entity_text = parts[1], parts[2]
                    tag_name = tag_info.split()[0]
                    # Solo consideramos entidades de morfología de neoplasia
                    if tag_name == "MORFOLOGIA_NEOPLASIA":
                        morfologias_set.add(entity_text.lower())
    return list(morfologias_set)

# ------------------------
# 2) Cargar textos para evaluación
# ------------------------
def load_cantemist_txt(base_dir="../data/cantemist"):
    records = []
    subsets = ["train-set", "dev-set1", "dev-set2", "test-set"]
    for subset in subsets:
        txt_folder = os.path.join(base_dir, subset, "cantemist-coding", "txt")
        if not os.path.isdir(txt_folder):
            continue
        for fname in os.listdir(txt_folder):
            if not fname.endswith(".txt"):
                continue
            path = os.path.join(txt_folder, fname)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            doc_id = f"{subset}/{fname}"
            records.append({"doc_id": doc_id, "text": text})
    return pd.DataFrame(records)

# ------------------------
# 3) Preparar lista de Document para la evaluación
# ------------------------
df_txt = load_cantemist_txt()
df_docs = [Document(page_content=row["text"], metadata={"doc_id": row["doc_id"]}) for _, row in df_txt.iterrows()]

# ------------------------
# 4) Extraer morfologías reales desde los .ann
# ------------------------
queries = extract_morfologias_from_ann()
print(f"Se encontraron {len(queries)} morfologías únicas para evaluación")
print("Ejemplos:", queries[:20])

# ------------------------
# 5) Evaluación RAG con Hit@k y Recall@k
# ------------------------
# ------------------------
# Evaluación RAG usando la lista de documentos ya cargados en `docs`
# ------------------------
# `docs` es la lista de Document que ya tienes en el retriever
# `queries` se extrae de los archivos .ann como antes

TOP_K = 10
hit_scores = []
recall_scores = []

for morph in tqdm(queries):
    try:
        # Recuperar hijos
        retrieved_children = retriever.invoke(morph)
        
        # Padres únicos
        retrieved_parent_ids = set(doc.metadata.get("source") for doc in retrieved_children if "source" in doc.metadata)
        
        # Padres relevantes
        relevant_parent_ids = set(doc.metadata.get("source") for doc in docs if morph in doc.page_content.lower() and "source" in doc.metadata)
        #print(retrieved_parent_ids)
        # Hit@k
        hit = int(bool(retrieved_parent_ids & relevant_parent_ids))
        hit_scores.append(hit)
        
        # Recall@k
        recall = len(retrieved_parent_ids & relevant_parent_ids) / len(relevant_parent_ids) if relevant_parent_ids else 0.0
        recall_scores.append(recall)
        
    except Exception as e:
        print(f"Error con query '{morph}': {e}")
        # Garantizar longitud consistente
        hit_scores.append(0)
        recall_scores.append(0.0)

# Verificar longitudes antes de crear DataFrame
print("Longitudes:", len(queries), len(hit_scores), len(recall_scores))
assert len(queries) == len(hit_scores) == len(recall_scores), "¡Las listas no tienen la misma longitud!"

# Crear DataFrame
df_metrics = pd.DataFrame({
    "morfología": queries,
    "Hit@k": hit_scores,
    "Recall@k": recall_scores
})

print(df_metrics.head(20))
print("Promedio Hit@k:", sum(hit_scores)/len(hit_scores))
print("Promedio Recall@k:", sum(recall_scores)/len(recall_scores))



Se encontraron 4110 morfologías únicas para evaluación
Ejemplos: ['lipomas', 'carcinoma de tipo lobulillar', 'estesioneuroblastoma gi-ii de hyams', 'astrocitoma granular de grado ii', 'rp pulmonar', 'ct3dn2m1', 'tumor neuroendocrino (tne) indiferenciado', 'carcinoma nasofaríngeo indiferenciado no queratinizante', 'células malignas de alto grado', 'metástasis de un carcinoma microcítico', 'lesión del segmento vi hepático', 'recidiva locorregional y a distancia de carcinoma de células de merkel', 'metástasis hepáticas de adenocarcinoma', 'tumor fibroso esclerosante de bajo grado', 'tumor germinal extragonadal mixto', 'adc endometrioide moderadamente diferenciado', 'carcinoma papilar invasivo', 'adenocarcinoma de páncreas estadio iv por metástasis', 'adenocarcinoma in situ, de tipo muccinoso', 'carcinoma de mama izquierdo en estadio metabólico m1']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4110/4110 [07:42<00:00,  8.89it/s]

Longitudes: 4110 4110 4110
                                           morfología  Hit@k  Recall@k
0                                             lipomas      0       0.0
1                        carcinoma de tipo lobulillar      1       1.0
2                 estesioneuroblastoma gi-ii de hyams      0       0.0
3                    astrocitoma granular de grado ii      0       0.0
4                                         rp pulmonar      0       0.0
5                                            ct3dn2m1      0       0.0
6           tumor neuroendocrino (tne) indiferenciado      0       0.0
7   carcinoma nasofaríngeo indiferenciado no quera...      1       1.0
8                      células malignas de alto grado      0       0.0
9              metástasis de un carcinoma microcítico      1       1.0
10                    lesión del segmento vi hepático      0       0.0
11  recidiva locorregional y a distancia de carcin...      0       0.0
12             metástasis hepáticas de adenocarcin

## Cómo funciona un **Parent Document Retriever**

El **Parent Document Retriever** es un tipo de **retriever jerárquico** que permite realizar búsquedas sobre documentos largos de manera más eficiente y granular, usando un enfoque de **padres e hijos**.

### 1. Concepto de padres e hijos

* **Documento padre**: representa un bloque grande de información, por ejemplo, un informe médico completo o un conjunto de párrafos extensos.
* **Documento hijo**: es un fragmento más pequeño extraído del documento padre (por ejemplo, 100-200 palabras o un párrafo), sobre el que realmente se calculan los embeddings y se realiza la búsqueda de similitud.

**Motivación**:
No siempre es eficiente generar embeddings de documentos completos muy grandes. Dividiendo en fragmentos hijos:

* Reducimos el tamaño de secuencia de los embeddings.
* Mejoramos la granularidad de la búsqueda.
* Permitimos que la recuperación se haga sobre porciones más relevantes del texto.

### 2. Flujo de recuperación

1. **Embeddings de hijos**:
   Cada hijo se transforma en un embedding usando un modelo de lenguaje (por ejemplo, PlanTL-GOB-ES).

2. **Indexación**:
   Los embeddings de los hijos se almacenan en un vectorstore (FAISS, Chroma, Milvus, etc.) para poder realizar búsquedas rápidas por similitud.

3. **Búsqueda por similitud**:
   Cuando se hace una consulta, se calcula el embedding del query y se busca en el vectorstore de hijos los más similares.

4. **Mapeo a padres**:
   Cada hijo pertenece a un padre. Una vez identificados los hijos más relevantes, se **recuperan los documentos padres completos** asociados.

5. **Refinamiento opcional**:
   En algunos casos, se pueden dividir los padres en frases o secciones para encontrar la parte exacta del documento que coincide mejor con la consulta.

### 3. Parámetros importantes

* `k`: número de **hijos** a considerar para la búsqueda principal.
* `fetch_k`: número de hijos adicionales que se evalúan para asegurar que se capturen los padres relevantes.
* `child_splitter` y `parent_splitter`: definen cómo se generan los hijos y padres a partir del texto completo.
* `search_type`: tipo de búsqueda en el vectorstore (por similitud por defecto).

### 4. Beneficios

* **Escalabilidad**: permite manejar documentos largos sin necesidad de embeddings gigantes.
* **Precisión**: se pueden encontrar secciones relevantes aunque el documento completo sea extenso.
* **Flexibilidad**: los hijos pueden ser generados a diferentes niveles (palabra, frase, párrafo), adaptándose a distintos casos de uso.

### 5. Ejemplo conceptual

```
Documento padre: Informe médico completo
├── Hijo 1: Introducción y antecedentes
├── Hijo 2: Hallazgos clínicos
├── Hijo 3: Resultados de laboratorio
└── Hijo 4: Conclusión y recomendaciones

Consulta: "leiomioma"
→ Se buscan los hijos más similares
→ Se devuelven los documentos padres correspondientes
```

> En resumen, un **Parent Document Retriever** permite usar embeddings de fragmentos pequeños para recuperar documentos completos, combinando eficiencia con precisión en búsquedas de texto largo.
